# Визуализация

Строим графики по рассчитанным визитам, регистрациям, конверсии и рекламным кампаниям. Все изображения сохраняются в `./charts` в формате PNG.


In [ ]:
%run ./ads.ipynb


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

charts_dir = Path("./charts")
charts_dir.mkdir(parents=True, exist_ok=True)

platform_order = ["web", "android", "ios"]

conversion_plot = conversion.copy()
conversion_plot["date_group"] = pd.to_datetime(conversion_plot["date_group"])
conversion_plot["conversion"] = conversion_plot["conversion"].replace(
    [float("inf"), float("-inf")], pd.NA
)

daily_metrics = (
    conversion_plot
    .groupby("date_group", as_index=False)
    .agg(
        visits=("visits", "sum"),
        registrations=("registrations", "sum"),
    )
    .sort_values("date_group")
)

daily_metrics["conversion"] = (
    daily_metrics["registrations"] / daily_metrics["visits"] * 100
)

ads_plot = ads_result.copy()
ads_plot["date_group"] = pd.to_datetime(ads_plot["date_group"])
ads_plot = (
    ads_plot
    .groupby("date_group", as_index=False)
    .agg(
        visits=("visits", "first"),
        registrations=("registrations", "first"),
        cost=("cost", "sum"),
    )
    .sort_values("date_group")
)

ads_campaigns = ads.copy()
ads_campaigns["date_group"] = pd.to_datetime(ads_campaigns["date"]).dt.normalize()
ads_campaigns = ads_campaigns.loc[
    ads_campaigns["date_group"].between(
        daily_metrics["date_group"].min(),
        daily_metrics["date_group"].max(),
    )
].copy()


In [ ]:
def format_date_axis(ax):
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)


def save_chart(fig, filename):
    fig.tight_layout()
    fig.savefig(charts_dir / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)


def campaign_periods(frame):
    periods = []

    for campaign, group in frame.groupby("utm_campaign"):
        dates = sorted(group["date_group"].drop_duplicates())

        if not dates:
            continue

        start = dates[0]
        previous = dates[0]

        for current in dates[1:]:
            if (current - previous).days > 1:
                periods.append((campaign, start, previous))
                start = current
            previous = current

        periods.append((campaign, start, previous))

    return periods


def add_campaign_spans(ax, frame):
    periods = campaign_periods(frame)
    campaigns = list(dict.fromkeys(campaign for campaign, _, _ in periods))
    cmap = plt.get_cmap("tab10")
    colors = {campaign: cmap(index % 10) for index, campaign in enumerate(campaigns)}

    for campaign, start, end in periods:
        ax.axvspan(
            start,
            end + pd.Timedelta(days=1),
            alpha=0.18,
            color=colors[campaign],
            label=campaign,
        )

    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    if unique:
        ax.legend(unique.values(), unique.keys(), loc="upper left", fontsize=8)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_metrics["date_group"], daily_metrics["visits"])
ax.set_title("Итоговые визиты")
ax.set_xlabel("Дата")
ax.set_ylabel("Визиты")
format_date_axis(ax)
save_chart(fig, "total_visits.png")


In [ ]:
visits_by_platform = (
    conversion_plot
    .pivot_table(
        index="date_group",
        columns="platform",
        values="visits",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(columns=platform_order, fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
visits_by_platform.plot(kind="bar", stacked=True, ax=ax, width=0.9)
ax.set_title("Визиты по платформам")
ax.set_xlabel("Дата")
ax.set_ylabel("Визиты")
ticks = range(0, len(visits_by_platform.index), 14)
ax.set_xticks(list(ticks))
ax.set_xticklabels(
    [visits_by_platform.index[index].strftime("%Y-%m-%d") for index in ticks],
    rotation=45,
    ha="right",
)
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Платформа")
save_chart(fig, "visits_by_platform.png")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_metrics["date_group"], daily_metrics["registrations"])
ax.set_title("Итоговые регистрации")
ax.set_xlabel("Дата")
ax.set_ylabel("Регистрации")
format_date_axis(ax)
save_chart(fig, "total_registrations.png")


In [ ]:
registrations_by_platform = (
    conversion_plot
    .pivot_table(
        index="date_group",
        columns="platform",
        values="registrations",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(columns=platform_order, fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
registrations_by_platform.plot(kind="bar", stacked=True, ax=ax, width=0.9)
ax.set_title("Регистрации по платформам")
ax.set_xlabel("Дата")
ax.set_ylabel("Регистрации")
ticks = range(0, len(registrations_by_platform.index), 14)
ax.set_xticks(list(ticks))
ax.set_xticklabels(
    [registrations_by_platform.index[index].strftime("%Y-%m-%d") for index in ticks],
    rotation=45,
    ha="right",
)
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Платформа")
save_chart(fig, "registrations_by_platform.png")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for platform in platform_order:
    platform_data = conversion_plot.loc[
        conversion_plot["platform"] == platform
    ].sort_values("date_group")

    ax.plot(
        platform_data["date_group"],
        platform_data["conversion"],
        label=platform,
    )

ax.set_title("Конверсия по платформам")
ax.set_xlabel("Дата")
ax.set_ylabel("Конверсия, %")
format_date_axis(ax)
ax.legend(title="Платформа")
save_chart(fig, "conversion_by_platform.png")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    daily_metrics["date_group"],
    daily_metrics["conversion"],
    label="Конверсия",
)

mean_conversion = daily_metrics["conversion"].mean()
ax.axhline(
    mean_conversion,
    linestyle="--",
    label=f"Средняя: {mean_conversion:.2f}%",
)

ax.set_title("Средняя конверсия")
ax.set_xlabel("Дата")
ax.set_ylabel("Конверсия, %")
format_date_axis(ax)
ax.legend()
save_chart(fig, "average_conversion.png")


In [ ]:
daily_ads_cost = (
    ads_plot
    .groupby("date_group", as_index=False)["cost"]
    .sum()
    .sort_values("date_group")
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_ads_cost["date_group"], daily_ads_cost["cost"])
ax.set_title("Стоимость рекламы")
ax.set_xlabel("Дата")
ax.set_ylabel("Затраты")
format_date_axis(ax)
save_chart(fig, "ads_cost.png")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    ads_plot["date_group"],
    ads_plot["visits"],
    label="Визиты",
)
add_campaign_spans(ax, ads_campaigns)
ax.set_title("Визиты и рекламные кампании")
ax.set_xlabel("Дата")
ax.set_ylabel("Визиты")
format_date_axis(ax)
save_chart(fig, "visits_with_ads.png")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    ads_plot["date_group"],
    ads_plot["registrations"],
    label="Регистрации",
)
add_campaign_spans(ax, ads_campaigns)
ax.set_title("Регистрации и рекламные кампании")
ax.set_xlabel("Дата")
ax.set_ylabel("Регистрации")
format_date_axis(ax)
save_chart(fig, "registrations_with_ads.png")


In [ ]:
sorted(path.name for path in charts_dir.glob("*.png"))
